In [ ]:
# ===============================
# 1. Environment Setup (Kaggle-safe)
# ===============================

# Install required packages (Kaggle: use --no-cache-dir to avoid OOM)
!pip install --no-cache-dir -U transformers datasets peft trl accelerate bitsandbytes


In [ ]:
# ===============================
# 2. Dataset Loading & Inspection
# ===============================

import json
import pandas as pd
from pathlib import Path

# Path to dataset
dataset_path = Path('../data/dataset.jsonl')

# Read JSONL dataset
samples = []
with open(dataset_path, 'r', encoding='utf-8') as f:
    for line in f:
        samples.append(json.loads(line))

# Convert to DataFrame for inspection
sample_df = pd.DataFrame(samples)
print('Columns:', sample_df.columns.tolist())
print('Sample rows:')
display(sample_df.head())

# Identify input/output columns
# Heuristics: prompt/description = input, code = output
input_col = None
output_col = None
for col in sample_df.columns:
    if 'prompt' in col or 'desc' in col:
        input_col = col
    if 'code' in col:
        output_col = col
print(f"Input column: {input_col}")
print(f"Output column: {output_col}")


In [ ]:
# ===============================
# 3. Preprocessing: Instruction-Response Formatting
# ===============================

from datasets import Dataset

# Use the columns detected above
def format_instruction(example):
    # SFTTrainer expects a dict with 'prompt' and 'completion' keys by default
    return {
        'prompt': example[input_col].strip(),
        'completion': example[output_col].strip(),
        'text': f"<s>[INST] {example[input_col].strip()} [/INST]\n{example[output_col].strip()}</s>"
    }

# Apply formatting
dataset = Dataset.from_pandas(sample_df)
dataset = dataset.map(format_instruction)

# Show a sample
print('Formatted sample:')
print(dataset[0])


In [ ]:
# ===============================
# 4. Model & Tokenizer Loading (Quantized)
# ===============================

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization config (bitsandbytes)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

# Load tokenizer
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# Load model (quantized)
print('Loading model (4-bit quantized)...')
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Enable gradient checkpointing for memory savings
model.gradient_checkpointing_enable()

# Enable flash attention if available
try:
    model.enable_flash_attn()
    print('Flash attention enabled!')
except Exception:
    print('Flash attention not available.')


In [ ]:
# ===============================
# 5. LoRA (QLoRA) & Trainer Setup
# ===============================

from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from transformers import TrainingArguments
import os

# LoRA config (QLoRA, memory efficient)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# Output dir for LoRA adapters
output_dir = "../outputs/mistral/lora"
os.makedirs(output_dir, exist_ok=True)

# Training arguments (Kaggle-safe)
training_args = TrainingArguments(
    per_device_train_batch_size=1,  # T4/P100 safe
    gradient_accumulation_steps=8,  # Effective batch size = 8
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    output_dir=output_dir,
    save_strategy="epoch",
    report_to="none",
    save_total_limit=1,
    max_steps=-1,  # Use -1 instead of None to avoid TypeError
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
)

# SFTTrainer for instruction tuning (must be defined before training loop)
# Use the default columns: 'prompt' and 'completion' (created in preprocessing)
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=training_args,
)


In [ ]:
# ===============================
# 6. Training Loop
# ===============================

# Clear CUDA cache before training to avoid OOM
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Train
trainer.train()

# Save LoRA adapter
trainer.model.save_pretrained(output_dir)
print(f"LoRA adapter saved to {output_dir}")


In [ ]:
# ===============================
# 7. Inference Test: Generate Manim Code
# ===============================

def generate_manim_code(prompt, model, tokenizer, max_new_tokens=512):
    model.eval()
    input_text = f"<s>[INST] {prompt.strip()} [/INST]"
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            eos_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the response (after [/INST])
    if '[/INST]' in decoded:
        decoded = decoded.split('[/INST]', 1)[-1].strip()
    return decoded

# Example prompt
test_prompt = "Create a Manim animation explaining Pythagoras theorem"
response = generate_manim_code(test_prompt, trainer.model, tokenizer)
print("Prompt:\n", test_prompt)
print("\nGenerated Manim code:\n", response)

# Save output for inspection
with open("../outputs/mistral/test_output.py", "w", encoding="utf-8") as f:
    f.write(response)
print("\nSample output saved to ../outputs/mistral/test_output.py")
